# Chapter 4 — Local Kernels on Learned Embeddings

Companion notebook to Chapter 4 of *Kernels and Transformers for Tabular Data*.

Reproduces:
- Figure 4.1: RBF affinity at $\sigma \in \{0.3, 1.0, 3.0\}$ on uniform 2D data.
- Figure 4.2: Self-tuning vs global RBF on density-disparate data.
- Figure 4.3: $\sigma$ histogram trajectory of `LearnedBandwidthRBF` with vs without regulariser.
- Figure 4.4: Tail comparison of RBF vs Cauchy at the same FWHM.
- Figure 4.5: Epanechnikov vs RBF affinity heatmap (intrinsic sparsity).

Worked example: Nadaraya–Watson regression on 1D toy with each kernel.

Runtime budget: < 5 minutes on CPU.

In [ ]:
import os
import time

import numpy as np
import torch
import matplotlib.pyplot as plt

from tabkernels.kernels.local import (
    RBFKernel,
    LearnedBandwidthRBF,
    MahalanobisKernel,
    CauchyKernel,
    EpanechnikovKernel,
)

torch.manual_seed(0)
np.random.seed(0)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

# Resolve figures dir robustly. Walk up from CWD looking for the
# `similarity-hierarchy-research/affinity/book/figures` directory.
def _find_figures_dir():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.basename(p) == 'similarity-hierarchy-research':
            return os.path.join(p, 'affinity', 'book', 'figures')
        p = os.path.dirname(p)
    # Fallback: relative to notebooks/ directory.
    return os.path.abspath(os.path.join('..', '..', 'affinity', 'book', 'figures'))

FIGURES_DIR = _find_figures_dir()
os.makedirs(FIGURES_DIR, exist_ok=True)
print('FIGURES_DIR:', FIGURES_DIR)

## Synthetic data

Two 2D datasets:
1. **Uniform Gaussian** (for §4.2): `N=80` points $\sim \mathcal{N}(0, I_2)$.
2. **Density-disparate clusters** (for §4.3): one tight cluster (scale $0.05$) and one wide cluster (scale $1.0$), 40 points each.

In [ ]:
torch.manual_seed(0)
X_uniform = torch.randn(80, 2)

torch.manual_seed(1)
tight = 0.05 * torch.randn(40, 2)
wide = torch.tensor([5.0, 5.0]) + 1.0 * torch.randn(40, 2)
X_disparate = torch.cat([tight, wide], dim=0)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].scatter(X_uniform[:, 0], X_uniform[:, 1], s=8)
axes[0].set_title('Uniform Gaussian (N=80)')
axes[0].set_aspect('equal')
axes[1].scatter(X_disparate[:40, 0], X_disparate[:40, 1], s=8, label='tight (σ=0.05)')
axes[1].scatter(X_disparate[40:, 0], X_disparate[40:, 1], s=8, label='wide (σ=1.0)')
axes[1].set_title('Density-disparate clusters')
axes[1].set_aspect('equal')
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

## Figure 4.1: RBF bandwidth sweep

Heatmap of the affinity matrix at $\sigma \in \{0.3, 1.0, 3.0\}$. Small $\sigma$ → near-identity; large $\sigma$ → near-uniform.

In [ ]:
sigmas = [0.3, 1.0, 3.0]
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, s in zip(axes, sigmas):
    k = RBFKernel(bandwidth=float(s))
    G = k(X_uniform, X_uniform).detach().cpu().numpy()
    im = ax.imshow(G, cmap='viridis', vmin=0, vmax=1)
    ax.set_title(rf'$\sigma = {s}$')
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.7)
fig.suptitle('Figure 4.1: RBF affinity at three bandwidths (uniform data)')
plt.savefig(os.path.join(FIGURES_DIR, 'fig_04_01_rbf_bandwidth.pdf'), bbox_inches='tight')
plt.show()

## Figure 4.2: Self-tuning vs global on density-disparate data

A single global $\sigma$ cannot serve clusters at very different scales. The self-tuning kernel adapts $\sigma_i$ per point and preserves the two-cluster block structure.

In [ ]:
k_global = RBFKernel(bandwidth=1.0)
k_self = RBFKernel(bandwidth='self_tuning', K=7)
G_global = k_global(X_disparate, X_disparate).detach().cpu().numpy()
G_self = k_self(X_disparate, X_disparate).detach().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(7, 3.2))
for ax, G, title in [
    (axes[0], G_global, 'Global RBF (σ=1.0)'),
    (axes[1], G_self, 'Self-tuning RBF (K=7)'),
]:
    im = ax.imshow(G, cmap='viridis', vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.7)
fig.suptitle('Figure 4.2: Single-bandwidth fails on multi-scale data')
plt.savefig(os.path.join(FIGURES_DIR, 'fig_04_02_self_tuning.pdf'), bbox_inches='tight')
plt.show()

## Figure 4.3: Learned-bandwidth $\sigma$ trajectory

Train `LearnedBandwidthRBF` for 200 steps on a Nadaraya–Watson loss. Without regularisation the encoder collapses $\sigma$ near zero on training points (memorisation); the regulariser $\lambda \mathbb{E}[\sigma_i^{-2}]$ keeps $\sigma$ bounded.

In [ ]:
torch.manual_seed(2)
X_train = torch.randn(64, 4)
y_train = (X_train[:, 0] > 0).float()

def train_track(reg, steps=200, lr=5e-2):
    torch.manual_seed(2)
    kernel = LearnedBandwidthRBF(d_in=4, hidden=16, regularizer=reg)
    opt = torch.optim.Adam(kernel.parameters(), lr=lr)
    sigma_log = []
    for t in range(steps):
        opt.zero_grad()
        G = kernel(X_train, X_train)
        row = G / G.sum(dim=1, keepdim=True).clamp_min(1e-8)
        yhat = row @ y_train
        loss = ((yhat - y_train) ** 2).mean()
        if reg > 0:
            loss = loss + kernel.regularization_loss(X_train)
        loss.backward()
        opt.step()
        if t in (0, 50, 100, 199):
            sigma_log.append((t, kernel._sigma(X_train).detach().clone()))
    return sigma_log

with_reg = train_track(reg=1e-1)
without_reg = train_track(reg=0.0)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.2), sharey=True)
for ax, log, title in [
    (axes[0], without_reg, 'No regulariser (σ collapses)'),
    (axes[1], with_reg, 'With λ=0.1 regulariser'),
]:
    for t, sigma in log:
        ax.hist(sigma.numpy(), bins=20, alpha=0.45, label=f'step {t}')
    ax.set_xlabel(r'$\sigma_i$')
    ax.set_title(title)
    ax.legend(fontsize=7)
    ax.set_yscale('log')
axes[0].set_ylabel('count')
fig.suptitle('Figure 4.3: Bandwidth-collapse risk and the regulariser remedy')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig_04_03_learned_sigma.pdf'), bbox_inches='tight')
plt.show()

## Figure 4.4: RBF vs Cauchy tail at matched FWHM

Both kernels are radially symmetric. Match the full-width-at-half-maximum so the cores agree, then plot $k(r)$ on a log scale to expose the heavy tail of Cauchy.

In [ ]:
# RBF FWHM = 2 sigma sqrt(ln 2). Cauchy nu=1, sigma_c: half-max at r = sigma_c.
sigma_rbf = 1.0
fwhm_rbf = 2 * sigma_rbf * np.sqrt(np.log(2))
# For Cauchy kernel k = (1 + r^2/nu)^{-(nu+1)/2}, with nu=1, half-max at r^2 = nu*(2^{2/(nu+1)} - 1) = 1.
# So FWHM_cauchy = 2 * sqrt(1) = 2 in those units. Rescale by alpha so 2*alpha = fwhm_rbf.
alpha = fwhm_rbf / 2.0

r = np.linspace(0, 6, 500)
k_rbf = np.exp(-(r ** 2) / (sigma_rbf ** 2))
k_cauchy = (1.0 + (r / alpha) ** 2) ** (-1.0)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))
axes[0].plot(r, k_rbf, label='RBF (Gaussian)')
axes[0].plot(r, k_cauchy, label=r'Cauchy ($\nu=1$)')
axes[0].axhline(0.5, color='gray', ls=':', lw=0.8)
axes[0].set_xlabel('r')
axes[0].set_ylabel('k(r)')
axes[0].set_title('Linear scale')
axes[0].legend()
axes[1].semilogy(r, np.maximum(k_rbf, 1e-12), label='RBF')
axes[1].semilogy(r, np.maximum(k_cauchy, 1e-12), label='Cauchy')
axes[1].set_xlabel('r')
axes[1].set_ylabel('k(r) (log)')
axes[1].set_title('Log scale: heavy tail visible')
axes[1].legend()
fig.suptitle('Figure 4.4: RBF vs Cauchy at matched FWHM')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig_04_04_rbf_vs_cauchy.pdf'), bbox_inches='tight')
plt.show()

## Figure 4.5: Epanechnikov intrinsic sparsity

Epanechnikov ($\max(0, 1 - r^2)$ with $r = \|x - x'\|/h$) is exactly zero beyond bandwidth $h$. Compare the affinity heatmap of an RBF and an Epanechnikov kernel on the same data; report the fraction of zero entries.

In [ ]:
torch.manual_seed(3)
X_compact = torch.randn(60, 4)
k_rbf = RBFKernel(bandwidth=1.0)
k_ep = EpanechnikovKernel(bandwidth=1.0)
G_rbf = k_rbf(X_compact, X_compact).detach().cpu().numpy()
G_ep = k_ep(X_compact, X_compact).detach().cpu().numpy()
zero_frac = float((G_ep == 0).mean())
print(f'Epanechnikov zero fraction: {zero_frac:.2%}')

fig, axes = plt.subplots(1, 2, figsize=(7, 3.2))
for ax, G, title in [
    (axes[0], G_rbf, 'RBF (h=1.0): dense'),
    (axes[1], G_ep, f'Epanechnikov (h=1.0): {zero_frac:.0%} zeros'),
]:
    im = ax.imshow(G, cmap='viridis', vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.7)
fig.suptitle('Figure 4.5: Compact support gives intrinsic sparsity')
plt.savefig(os.path.join(FIGURES_DIR, 'fig_04_05_epanechnikov_sparsity.pdf'), bbox_inches='tight')
plt.show()

## Worked example: Nadaraya–Watson with each kernel on a 1D regression

Target: $y = \sin(3 x) + 0.1 \varepsilon$ on $x \in [-2, 2]$. Train on $N = 50$ points, evaluate MSE on a dense test grid for each kernel.

In [ ]:
torch.manual_seed(4)
x_train = torch.linspace(-2, 2, 50).unsqueeze(1)
y_train_1d = torch.sin(3 * x_train.squeeze()) + 0.1 * torch.randn(50)
x_test = torch.linspace(-2, 2, 200).unsqueeze(1)
y_test = torch.sin(3 * x_test.squeeze())

kernels_1d = {
    'RBF (σ=0.3)': RBFKernel(bandwidth=0.3),
    'RBF self-tuning (K=5)': RBFKernel(bandwidth='self_tuning', K=5),
    'Cauchy (ν=1)': CauchyKernel(nu=1.0),
    'Epanechnikov (h=0.6)': EpanechnikovKernel(bandwidth=0.6),
    'Mahalanobis (d=1)': MahalanobisKernel(d_in=1),
}
rows = []
for name, k in kernels_1d.items():
    if name.startswith('RBF self-tuning'):
        # cross-Gram with self-tuning needs careful handling: build on train then
        # query against train sigma. Simplify by using a fixed bandwidth heuristic.
        with torch.no_grad():
            sigma_tr = k._self_tuning_sigma(x_train)  # noqa: SLF001
            D2 = torch.cdist(x_test, x_train).pow(2)
            # Use sigma_q = sigma_train.median() as a stand-in for the test points.
            sigma_q = sigma_tr.median().expand(x_test.shape[0])
            sigma_ij = sigma_q.unsqueeze(1) * sigma_tr.unsqueeze(0)
            G = torch.exp(-D2 / sigma_ij.clamp_min(1e-12))
    else:
        with torch.no_grad():
            G = k(x_test, x_train)
    row = G / G.sum(dim=1, keepdim=True).clamp_min(1e-8)
    yhat = row @ y_train_1d
    mse = ((yhat - y_test) ** 2).mean().item()
    rows.append((name, mse))

print(f'{"Kernel":<25s} | MSE')
print('-' * 40)
for name, mse in rows:
    print(f'{name:<25s} | {mse:.4f}')

## Exercises

1. **Biweight from EpanechnikovKernel.** Build `EpanechnikovKernel(bandwidth=0.6, power=2)` and verify intrinsic sparsity matches the `power=1` variant (compact support is the same; only the *shape* inside the support changes).
2. **Anisotropic Mahalanobis.** Generate data where one coordinate is irrelevant ($x_2 \sim \mathcal{N}(0, 4)$ unrelated to $y$) and train `MahalanobisKernel` to ignore it; visualise the learned $A$.
3. **Runtime: RBF vs Cauchy.** Time the forward pass at $N = 1000$, $d = 16$. Both should be dominated by `cdist`; the kernel evaluation itself is negligible.


In [ ]:
# Exercise 1
torch.manual_seed(5)
X_e = torch.randn(40, 4)
k1 = EpanechnikovKernel(bandwidth=0.6, power=1)
k2 = EpanechnikovKernel(bandwidth=0.6, power=2)
z1 = (k1(X_e, X_e) == 0).float().mean().item()
z2 = (k2(X_e, X_e) == 0).float().mean().item()
print(f'Epanechnikov zero frac: {z1:.2%}, biweight zero frac: {z2:.2%} (should match)')

# Exercise 3: timing
torch.manual_seed(6)
X_big = torch.randn(1000, 16)
k_rbf_big = RBFKernel(bandwidth=1.0)
k_cauchy_big = CauchyKernel(nu=1.0)
for label, k in [('RBF', k_rbf_big), ('Cauchy', k_cauchy_big)]:
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = k(X_big, X_big)
    print(f'{label}: {1000 * (time.perf_counter() - t0):.2f} ms')